# Workshop, Part 3: A Real Model from Hugging Face

**Artificial Intelligence and Machine Learning · Transformers workshop**

**Hugging Face** is a public library of trained models: more than a million of them, free to download. You do not have to train a model yourself to use one.

Today we download a small language model, **SmolLM2-135M-Instruct**, and talk to it. It runs on your laptop, without an internet connection once it is downloaded.

| | Your name generator | SmolLM2-135M |
|---|---|---|
| Parameters | about 19,000 | about 135,000,000 |
| Learned from | one list of names | a very large collection of text |
| Training | seconds on a laptop | many graphics cards, for a long time |
| Reads | single letters | **tokens** (pieces of words) |
| Design | Transformer blocks with attention | Transformer blocks with attention |

Run the cells from top to bottom with **Shift+Enter**. Cells marked **Your turn** ask you to change something.

## 1. Load the model

One line downloads the model (the first time only) and prepares it. `pipeline` is the simplest way to use a Hugging Face model: you say what kind of task you want and which model to use.

In [ ]:
import warnings

warnings.filterwarnings("ignore")                 # hide technical messages

import transformers
from transformers import pipeline

transformers.logging.set_verbosity_error()

MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"
chat = pipeline("text-generation", model=MODEL)
print("Model loaded:", MODEL)

## 2. Ask a question

A chat model receives a list of messages. Each message has a **role** (`user` is you) and a **content** (your text). The small function `ask` wraps this so that you can simply type a question.

In [ ]:
def ask(question, temperature=0.7, max_new_tokens=80, system=None):
    # Send one question to the model and return its answer as text.
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": question})
    result = chat(messages, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature)
    return result[0]["generated_text"][-1]["content"].strip()


print(ask("Explain in two sentences what a neural network is."))

**Your turn.** Change the question and run the cell again. Try two or three different questions.

In [ ]:
print(ask("Write a two-line poem about Vilnius."))

## 3. The same design, about 7,000 times larger

Your name generator and this model are built from the same kind of blocks. The main difference is the numbers.

In the first line, replace `19_231` by the number of parameters your own notebook printed in Part 1 (you wrote it on page 1 of the guide). The underscore only makes a long number easier to read.

In [ ]:
YOUR_PARAMETERS = 19_231          # <-- the number your notebook printed in Part 1

config = chat.model.config
its_parameters = sum(p.numel() for p in chat.model.parameters())

print(f"{'':<28}{'Your model':>14}{'SmolLM2-135M':>16}")
print(f"{'Parameters':<28}{YOUR_PARAMETERS:>14,}{its_parameters:>16,}")
print(f"{'Transformer blocks':<28}{2:>14}{config.num_hidden_layers:>16}")
print(f"{'Attention heads per block':<28}{2:>14}{config.num_attention_heads:>16}")
print(f"{'Length of each vector':<28}{32:>14}{config.hidden_size:>16}")
print(f"{'Size of the vocabulary':<28}{'about 30':>14}{config.vocab_size:>16,}")
print(f"\nThis model is about {its_parameters / YOUR_PARAMETERS:,.0f} times larger than yours.")

## 4. Tokens instead of letters

Your model read one **letter** at a time. Large models read **tokens**: frequent words are one token, rare words are cut into several pieces. The table in Step 3 showed how many different tokens this model knows.

The symbol `_` below stands for a space at the start of a token. `[part]` means that a single letter was cut into two tokens. This can happen with letters that are rare in English text, such as `ę` or `š`.

In [ ]:
def show_tokens(text):
    ids = chat.tokenizer.encode(text, add_special_tokens=False)
    pieces = [chat.tokenizer.decode([i]).replace(" ", "_") for i in ids]
    # A letter such as ę can be split into two tokens; half a letter cannot be printed.
    pieces = ["[part]" if "\ufffd" in piece else piece for piece in pieces]
    print(f"{text!r}  ->  {len(ids)} tokens")
    print("   " + " | ".join(pieces))


show_tokens("The cat sat on the mat.")
show_tokens("Transformers are surprisingly simple.")
show_tokens("Kęstutis lives in Vilnius.")

**Your turn.** Replace the text between the quotes with your own first name and surname. Is it one token or several? Common English words are usually one token; names from smaller languages are cut into many pieces.

In [ ]:
show_tokens("Your Name")

## 5. Temperature again

In Part 1 you wrote `pick_next`, which divides the scores by the temperature. This model has exactly the same control. Low temperature: careful and repetitive. High temperature: varied and sometimes strange.

In [ ]:
question = "Invent a name for a coffee shop. Answer with the name only."

for temperature in [0.2, 0.8, 1.4]:
    print(f"Temperature {temperature}:")
    for _ in range(3):
        print("   ", ask(question, temperature=temperature, max_new_tokens=12))

## 6. Give the model a role

A **system** message is an instruction that comes before your question. It tells the model how to behave. Products built on language models use this all the time.

In [ ]:
print(ask("How is the weather today?",
          system="You are a pirate. Answer in one short sentence, like a pirate."))

**Your turn.** Invent your own role (a strict teacher, a poet, a football commentator) and ask the same question.

In [ ]:
print(ask("How is the weather today?",
          system="You are ..."))

## 7. Find its limits

This model writes fluent English, but it is small. **Fluent does not mean correct.** Run the three tests and write the results in the table of your printed guide.

1. A fact. 2. A calculation. 3. The task of your own model.

In [ ]:
print("1.", ask("What is the capital of Lithuania? Answer in one sentence.", temperature=0.3))
print()
print("2.", ask("What is 17 times 23? Answer with the number only.", temperature=0.3, max_new_tokens=15))
print()
print("3.", ask("Invent five new Lithuanian first names. Answer with the names only.", max_new_tokens=40))

**Think about it.**

- Which answers were right? Which were wrong but sounded confident?
- For task 3, which model did better: this one, or your 19,000-parameter name generator? A small model trained for **one** task can beat a much larger general model at that task.

A model that states something false in a confident tone is said to **hallucinate**. Larger models do it less often, but they all do it.

## 8. For your group project

**Finding models.** Go to `huggingface.co/models` and filter by task (text classification, translation, image classification, ...). Every model page shows the code to load it. Check the size before you download: this one is 270 MB, many others are several gigabytes.

**Other tasks work the same way.** For example, a model that decides whether a sentence is positive or negative (this downloads another 270 MB, so run it only if you have time and a good connection):

In [ ]:
# Optional: remove the # from the next three lines to try a sentiment model.
# sentiment = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
# print(sentiment("This workshop was much better than I expected."))
# print(sentiment("My code does not run and I am tired."))

**Using a Hugging Face model in your app.** In `app.py`, load the model once and reuse it:

```python
import streamlit as st
from transformers import pipeline

@st.cache_resource            # load once, keep in memory
def load_model():
    return pipeline("text-generation", model="HuggingFaceTB/SmolLM2-135M-Instruct")

chat = load_model()
question = st.text_input("Ask something")
if question:
    answer = chat([{"role": "user", "content": question}], max_new_tokens=80)
    st.write(answer[0]["generated_text"][-1]["content"])
```

Add `transformers` to `requirements.txt`. Keep the model small: the free hosting has limited memory.

**Remember the project rule:** the model you present must be one of the models discussed in the course. A downloaded model can be your baseline or your comparison, and you must be able to explain what it does.